In [ ]:
from transformers import AutoTokenizer
import json
import os
import random
random.seed(42)

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

#tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-xl") #BPE tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased") #WordPiece tokenizer
#tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased") #Unigram tokenizer

#t_label = "BPE"
t_label = "WordPiece"
#t_label = "Unigram"

real_path = "./distribution/part_"
tokenized_path = "./distribution/tokenized/part_"

def walk_directory(directory):
    l = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                l.append(os.path.join(root, file))
    return l

def create_article_list(file):
    #print(f"Creating article list from {file}")
    articles = []
        # Load the JSON file
    with open(file, "r") as fl:
        f = json.load(fl)
        articles.append(f["original_doc"])
        articles.append(f['articles']['gpt4o']['article'])
        articles.append(f['articles']['claude3.5sonnet']['article'])
        articles.append(f['articles']['llama3.1-405b']['article'])
        articles.append(f['articles']['qwen1.5-110b']['article'])
    #print(f"Found {len(articles)} articles in {file}")
    return articles

def get_topic(path):
    topic =''
    with open(path, "r") as f:
        data = json.load(f)
        topic = data["topic"]
    return topic

In [47]:
tmp = "Hello World! This is a test."
print(tokenizer.tokenize(tmp))

['Hello', 'World', '!', 'This', 'is', 'a', 'test', '.']


In [3]:
def tokenize_article(path):
    token_distribution = {}
    topic = get_topic(path)
    for article in create_article_list(path):
            # Tokenize the article
            #print(f"Tokenizing article: {article[:50]}...")
            article = article.strip()
            article = article.lower()
            data = tokenizer(article, return_tensors="pt", truncation=True, max_length=512)#tokenizer.model_max_length)
    for token_id in data["input_ids"][0]:
            token = tokenizer.decode(token_id.item())
            if token not in token_distribution:
                token_distribution[token] = {"count": 1, "topic": [topic]}
            else:
                token_distribution[token]["count"] += 1
                if topic not in token_distribution[token]["topic"]:
                    token_distribution[token]["topic"].append(topic)
    return token_distribution

In [4]:
def create_distribution(piecelist):
    final_token_distribution = {}
    for file in piecelist:
        print(f"Processing file: {file}")
        token_distribution = tokenize_article(file)
        # Merge the token distributions
        for item in token_distribution:
            if item not in final_token_distribution:
                final_token_distribution[item] = token_distribution[item]
            else:
                final_token_distribution[item]["count"] += token_distribution[item]["count"]
                for topic in token_distribution[item]["topic"]:
                    if topic not in final_token_distribution[item]["topic"]:
                        final_token_distribution[item]["topic"].append(topic)
    sorted_distribution = sorted(final_token_distribution.items(), key=lambda x: x[1]["count"], reverse=True)
    output_file = f"./json_files/new_tokenizer_tests/new_token_distribution_{t_label}.json"
    os.remove(output_file) if os.path.exists(output_file) else None
    # Save the final token distribution to a JSON file
    with open(output_file, "w") as f:
        json.dump(sorted_distribution, f, indent=4)

In [50]:
for i in range(0,10):
    piece = walk_directory(f"{real_path}{i}")
    token_distribution = create_distribution(piece)

Processing file: ./distribution/real/part_0\0000.json
Processing file: ./distribution/real/part_0\0010.json
Processing file: ./distribution/real/part_0\0020.json
Processing file: ./distribution/real/part_0\0030.json
Processing file: ./distribution/real/part_0\0040.json
Processing file: ./distribution/real/part_0\0050.json
Processing file: ./distribution/real/part_0\0060.json
Processing file: ./distribution/real/part_0\0070.json
Processing file: ./distribution/real/part_0\0080.json
Processing file: ./distribution/real/part_0\0090.json
Processing file: ./distribution/real/part_0\0100.json
Processing file: ./distribution/real/part_0\0110.json
Processing file: ./distribution/real/part_0\0120.json
Processing file: ./distribution/real/part_0\0130.json
Processing file: ./distribution/real/part_0\0140.json
Processing file: ./distribution/real/part_0\0150.json
Processing file: ./distribution/real/part_0\0160.json
Processing file: ./distribution/real/part_0\0170.json
Processing file: ./distribut

In [4]:
# create a jsoin file that contains all tokens that appear at most 2 times in two or more tokenizers
# open the three json files with the token distributions and load them
with open(f"./json_files/new_tokenizer_tests/new_token_distribution_BPE.json", "r") as f:
    bpe_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/new_token_distribution_WordPiece.json", "r") as f:
    wp_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/new_token_distribution_Unigram.json", "r") as f:
    unigram_distribution = json.load(f)

# create a new json file with the tokens that appear at most 2 times between the three distributions
common_tokens = {}
for token, data in bpe_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "BPE"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "BPE" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", BPE"
for token, data in wp_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "WordPiece"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "WordPiece" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", WordPiece"
for token, data in unigram_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "Unigram"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "Unigram" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", Unigram"
# Save the common tokens to a JSON file
output_common_file = "./json_files/new_tokenizer_tests/common_tokens.json"
os.remove(output_common_file) if os.path.exists(output_common_file) else None   
with open(output_common_file, "w") as f:
    json.dump(common_tokens, f, indent=4)
    print(f"Common tokens saved to {output_common_file}")

Common tokens saved to ./json_files/new_tokenizer_tests/common_tokens.json


In [8]:
#count the topics in the common tokens and compare them to all the topics from the , token in one distribution
topics_count = {}
for token, data in common_tokens.items():
    for topic in data["topic"]:
        if topic not in topics_count:
            topics_count[topic] = 1
        else:
            topics_count[topic] += 1
print("Topics count in common tokens:")
for topic, count in topics_count.items():
    print(f"{topic}: {count}")

print("\n\n")
#print the number of common tokens
print(f"Number of common tokens: {len(common_tokens)}")
#sum all the counts of the topics to get the average number of topics per token
total_topics = sum(len(data["topic"]) for data in common_tokens.values())
average_topics = total_topics / len(common_tokens) if common_tokens else 0
print(f"Average number of topics per token: {average_topics:.2f}")

Topics count in common tokens:
Company Policies: 465
Local Education Systems: 774
Incident Report: 1055
Regional Folklore and Myths: 829
Neighborhood Stories: 860
Local News: 838
Local Politics and Governance: 580
Regional Cuisine and Recipes: 1500
Local Technology and Innovation: 611
Small and Medium Enterprises: 694
Local Economy and Market: 916
News Stories: 836
Local Sports and Activities: 1087
Local Environmental Issues: 855
Local Health and Wellness: 915
Cybersecurity News: 1016
Local Arts and Culture: 533



Number of common tokens: 11252
Average number of topics per token: 1.28


In [ ]:
#choose a random article from all in the distribution folder
def choose_random_article():
    real_path = "./distribution/real/"
    all_files = walk_directory(real_path)
    random_file = random.choice(all_files)
    return random_file

#select a random article and extract its topic. Based on the topic, select a list of 10 randomm tokens from the common tokens that are related to the topic
def select_random_tokens_for_topic(topic, common_tokens, num_tokens=10):
    related_tokens = [token for token, data in common_tokens.items() if topic in data["topic"]]
    if len(related_tokens) < num_tokens:
        print(f"Not enough related tokens for topic '{topic}'. Found: {len(related_tokens)}")
        return related_tokens
    return random.sample(related_tokens, num_tokens)

def main():
    random_article = choose_random_article()
    topic = get_topic(random_article)
    print(f"Selected article path: {random_article}")
    print(f"Randomly selected article topic: {topic}")
    
    common_tokens_file = "./json_files/new_tokenizer_tests/common_tokens.json"
    with open(common_tokens_file, "r") as f:
        common_tokens = json.load(f)
    selected_tokens = select_random_tokens_for_topic(topic, common_tokens)
    print(f"Selected tokens related to topic '{topic}': {selected_tokens}")
    #save the key facts and the additional facts from the article in two varibles
    with open(random_article, "r") as f:
        article_data = json.load(f)
        key_facts = article_data.get("key_facts", [])
        other_facts = article_data.get("other_facts", [])
        #choose 5 random other facts
        if len(other_facts) > 5:
            other_facts = random.sample(other_facts, 5)
        else:
            print(f"Not enough other facts. Found: {len(other_facts)}")
    print(f"Key facts: {key_facts}")
    print(f"Other facts: {other_facts}")

    kf_prompt = ", ".join(key_facts)
    of_prompt = ", ".join(other_facts)
    st_prompt = ", ".join(selected_tokens)

    prompt = f"""You are an AI assistant tasked with generating an article based on the following key facts (included in <keyfacts></keyfacts> tags) and additional facts (included in <additionalfacts></additionalfacts> tags). Use the provided tokens (included in <tokens></tokens> tags) as much as you can to enhance the article's content and ensure it is relevant to the topic '{topic}'.
Key Facts: <keyfacts>{kf_prompt}</keyfacts>
Additional Facts: <additionalfacts>{of_prompt}</additionalfacts>
Tokens to use: <tokens>{st_prompt}</tokens>
Please generate a coherent and informative article that incorporates these elements."""
    return prompt

if __name__ == "__main__":
    prompt = main()
    print("\nGenerated Prompt:")
    print(prompt)
    with open("./json_files/new_tokenizer_tests/generated_prompt.txt", "w") as f:
        f.write(prompt)

Selected article path: ./distribution/real/part_4\0194.json
Randomly selected article topic: Cybersecurity News
Selected tokens related to topic 'Cybersecurity News': ['hoping', ' flagged', ' endowed', ' racing', 'fraudulent', ' deceive', ' tactic', '##pts', ' mic', 'tearing']
Key facts: ['The security of video conferencing software is critical in remote work to protect communications among remote teams.', 'Video conferencing platforms like Zoom, Microsoft Teams, and Google Meet are widely used with increased security concerns.', 'Key vulnerabilities in video conferencing include unauthorized access, data interception, and identity theft.', 'End-to-end encryption ensures that data is only decrypted at the endpoints, though difficult to implement for video conferencing.', 'Access control through authentication and authorization is crucial in securing video conferences.']
Other facts: ['Tech companies like Google and Microsoft are enhancing encryption measures for their conferencing tool